<a href="https://colab.research.google.com/github/tipismaca/Trabajo-Bebidas-del-Maipo-SA/blob/main/Codigo/Entrega2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Caso de Estudio: Bebidas del Maipo S.A. — Entrega 2
**Autoras:** Maytte Almuna, Macarena Campos, Darlyn Jerez y Valentina Roa


## 1. Configuración inicial
Clonación del repositorio y librerías

In [ ]:
# Clona el repositorio para tener acceso a los datos
!rm -rf Trabajo-Bebidas-del-Maipo-SA
!git clone https://github.com/tipismaca/Trabajo-Bebidas-del-Maipo-SA.git

Cloning into 'Trabajo-Bebidas-del-Maipo-SA'...
remote: Enumerating objects: 514, done.
remote: Counting objects: 100% (160/160), done.
remote: Compressing objects: 100% (140/140), done.
remote: Total 514 (delta 68), reused 60 (delta 16), pack-reused 354 (from 1)
Receiving objects: 100% (514/514), 41.67 MiB | 20.52 MiB/s, done.
Resolving deltas: 100% (261/261), done.


In [ ]:
%pip install -q "pulp>=2.7,<3"   # PuLP 2.x trae CBC incluido

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pulp
from IPython.display import display


## 2. Carga de datos

In [ ]:
ruta_base = "Trabajo-Bebidas-del-Maipo-SA/Datos_crudos/"
ruta_resultados = "Trabajo-Bebidas-del-Maipo-SA/Resultados/"



---
## Parte A1. Balance de inventario por sku y semana.


In [ ]:
# =====================================================================
# 2.1 DATOS CRUDOS DEL HORIZONTE TÁCTICO
# =====================================================================
maestro      = pd.read_csv(ruta_base + "maestro_sku.csv")
inv_inicial  = pd.read_csv(ruta_base + "inventario_inicial.csv")
capacidad    = pd.read_csv(ruta_base + "capacidad_horizonte.csv")
setups       = pd.read_csv(ruta_base + "setups_cip.csv")
compromisos  = pd.read_csv(ruta_base + "compromisos_semana_focal.csv")

# =====================================================================
# 2.2 TABLAS DE TRASPASO DE LA ENTREGA 1 (Parte C)
# =====================================================================
tabla_c1 = pd.read_csv(ruta_resultados + "merma_por_linea.csv")
tabla_c2 = pd.read_csv(ruta_resultados + "sobrellenado_por_formato.csv")
tabla_c3 = pd.read_csv(ruta_resultados + "pronostico_sku_horizonte.csv")
tabla_c4 = pd.real_csv(ruta_resultados + "sigma_error_sku_horizonte.csv")

# C.1: ρ_ℓ por línea
rho = dict(zip(tabla_c1["Línea"], tabla_c1["ρ (proporción)"].astype(float)))

# C.2: litros de jarabe adicionales por caja, por formato (se usa en A3)
jarabe_extra_L_caja = dict(zip(tabla_c2["Formato (L)"].astype(float),
                               tabla_c2["Jarabe adicional (L/caja)"].astype(float)))

# C.3 y C.4: matriz SKU × (h=1..h=13) a formato largo (sku, h, valor)
def matriz_a_largo(tabla, nombre_valor):
    largo = tabla.melt(id_vars=["Familia", "SKU"], var_name="h", value_name=nombre_valor)
    largo["h"] = largo["h"].str.replace("h=", "", regex=False).astype(int)
    largo[nombre_valor] = largo[nombre_valor].astype(float)
    return largo.rename(columns={"SKU": "sku", "Familia": "familia"})

pronostico = matriz_a_largo(tabla_c3, "d_hat")
sigma_err  = matriz_a_largo(tabla_c4, "sigma")

# =====================================================================
# 2.3 VALIDACIONES DE CONSISTENCIA ENTRE E1 Y LOS DATOS DE E2
# =====================================================================
SKUS   = maestro["sku"].tolist()
LINEAS = sorted(maestro["linea_envasado"].unique())
T      = sorted(capacidad["t"].astype(int).tolist())

assert set(rho) == set(LINEAS), f"C.1 no trae ρ para todas las líneas: {rho}"
assert all(0 <= r < 1 for r in rho.values()), "ρ_ℓ debe estar en [0, 1)."
assert set(jarabe_extra_L_caja) == set(maestro["formato_L"]), "C.2 no cubre todos los formatos."
for nombre, df in [("C.3", pronostico), ("C.4", sigma_err)]:
    assert set(df["sku"]) == set(SKUS), f"{nombre}: los SKU no coinciden con maestro_sku."
    assert set(df["h"]) == set(T), f"{nombre}: los horizontes no son {T[0]}..{T[-1]}."
    assert not df.duplicated(["sku", "h"]).any(), f"{nombre}: hay (sku, h) duplicados."
assert (pronostico["d_hat"] >= 0).all(), "Hay pronósticos negativos en C.3."
assert (sigma_err["sigma"] >= 0).all(), "Hay σ̂ negativos en C.4."
assert set(inv_inicial["sku"]) == set(SKUS), "inventario_inicial no cubre los 9 SKU."

# El horizonte h del pronóstico corresponde a la semana t = h del plan
# (h = 1 es el lunes 28/09/2026, primera fila de capacidad_horizonte.csv).
d_hat = {(r.sku, r.h): r.d_hat for r in pronostico.itertuples()}

# Aviso: pronósticos que superan 3 veces la mediana del SKU en el horizonte
mediana_sku = pronostico.groupby("sku")["d_hat"].transform("median")
picos = pronostico[pronostico["d_hat"] > 3 * mediana_sku].merge(
    capacidad[["t", "fecha_lunes"]], left_on="h", right_on="t")
if len(picos):
    picos["veces_mediana"] = (picos["d_hat"]
                              / picos["sku"].map(pronostico.groupby("sku")["d_hat"].median())).round(1)
    print("⚠ Pronósticos atípicos (> 3 × mediana del SKU):")
    display(picos[["sku", "h", "fecha_lunes", "d_hat", "veces_mediana"]])

print(f"SKU: {len(SKUS)} | Líneas: {LINEAS} | Semanas: {T[0]}–{T[-1]} "
      f"({capacidad['fecha_lunes'].iloc[0]} a {capacidad['fecha_lunes'].iloc[-1]})")
print("ρ_ℓ:", rho)
print("Jarabe extra (L/caja):", jarabe_extra_L_caja)
print(f"Demanda pronosticada total del horizonte: {sum(d_hat.values()):,.0f} cajas")

In [ ]:
# =====================================================================
# PARTE A1 — BALANCE DE INVENTARIO POR SKU Y SEMANA (con backlogging y merma)
#   s[i,t-1] + (1-ρ_ℓ(i))·x[i,t] + b[i,t] = d[i,t] + s[i,t] + b[i,t-1]
# =====================================================================

# ---------------------------------------------------------------------
# 1. DATOS
# ---------------------------------------------------------------------
datos = maestro.set_index("sku").join(inv_inicial.set_index("sku"))

linea_de = datos["linea_envasado"].to_dict()                                # ℓ(i)
p_var    = datos["costo_variable_caja_clp"].astype(float).to_dict()         # p^i
h        = datos["costo_inventario_caja_semana_clp"].astype(float).to_dict()  # h^i
pi_atr   = datos["penalidad_atraso_caja_semana_clp"].astype(float).to_dict()  # π^i
s0       = datos["inventario_inicial_cajas"].astype(float).to_dict()        # s^i_0
b0       = {i: 0.0 for i in SKUS}                                           # b^i_0 (supuesto)
d        = d_hat                                                            # d^i_t
rend     = {i: 1 - rho[linea_de[i]] for i in SKUS}                          # 1 - ρ_ℓ(i)

tabla_a1_1 = pd.DataFrame({
    "SKU": SKUS,
    "Línea": [linea_de[i] for i in SKUS],
    "ρ_ℓ": [rho[linea_de[i]] for i in SKUS],
    "1 - ρ_ℓ": [rend[i] for i in SKUS],
    "s_0 (cajas)": [s0[i] for i in SKUS],
    "Σ d_t (cajas)": [sum(d[i, t] for t in T) for i in SKUS],
    "p (CLP/caja)": [p_var[i] for i in SKUS],
    "h (CLP/caja·sem)": [h[i] for i in SKUS],
    "π (CLP/caja·sem)": [pi_atr[i] for i in SKUS],
})
print("Tabla A1.1 — Datos del balance por SKU")
display(tabla_a1_1.round(6))

# ---------------------------------------------------------------------
# 2. MODELO, VARIABLES Y BALANCE
# ---------------------------------------------------------------------
# Las partes A2–A8 agregan restricciones a este mismo `modelo` y
# términos nuevos al diccionario `costos`.
modelo = pulp.LpProblem("BdM_LotSizing", pulp.LpMinimize)

x = pulp.LpVariable.dicts("x", (SKUS, T), lowBound=0)  # producción bruta
s = pulp.LpVariable.dicts("s", (SKUS, T), lowBound=0)  # inventario final
b = pulp.LpVariable.dicts("b", (SKUS, T), lowBound=0)  # backlog final

for i in SKUS:
    for t in T:
        s_prev = s0[i] if t == 1 else s[i][t - 1]
        b_prev = b0[i] if t == 1 else b[i][t - 1]
        modelo += (
            s_prev + rend[i] * x[i][t] + b[i][t] == d[i, t] + s[i][t] + b_prev,
            f"balance_{i}_{t}",
        )

costos = {
    "Producción (p·x)": pulp.lpSum(p_var[i] * x[i][t] for i in SKUS for t in T),
    "Inventario (h·s)": pulp.lpSum(h[i] * s[i][t] for i in SKUS for t in T),
    "Atraso (π·b)":     pulp.lpSum(pi_atr[i] * b[i][t] for i in SKUS for t in T),
}
modelo += pulp.lpSum(costos.values())

print(f"\nRestricciones de balance: {len(modelo.constraints)} "
      f"(= {len(SKUS)} SKU × {len(T)} semanas)")
print(f"Variables: {len(modelo.variables())} (x, s, b)")
